# Machine Learning Finetuning

## Import Modules

In [10]:
# Make sure you are at the parent directory
from pathlib import Path
import sys
import os

# Define MODE
MODE = "LOCAL" # Either "COLAB", "LOCAL", "EXPANSE"
if MODE.upper() not in ('EXPANSE', 'COLAB', 'LOCAL'):
    raise Exception("Invalid mode, the only acceptible are 'EXPANSE', 'COLAB', 'LOCAL'")
if MODE.upper() == "COLAB":
    from google.colab import drive
    drive.mount('/content/drive')

# --- Java 17 for PySpark (LOCAL) ---
# PySpark 4.x requires Java 17/21. On a fresh Mac the default `java` may be older
# (e.g. 15), and VS Code notebook kernels often don't inherit JAVA_HOME from your
# shell -- which causes `JAVA_GATEWAY_EXITED` / UnsupportedClassVersionError when
# create_spark_session() launches the JVM. Point Spark at a JDK 17 here.
if MODE.upper() == "LOCAL" and not os.environ.get("JAVA_HOME"):
    _java_candidates = [
        "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",  # Homebrew (Apple Silicon)
        "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",     # Homebrew (Intel)
        "/Library/Java/JavaVirtualMachines/temurin-17.jdk/Contents/Home",  # Temurin / Adoptium
    ]
    for _jh in _java_candidates:
        if Path(_jh).exists():
            os.environ["JAVA_HOME"] = _jh
            os.environ["PATH"] = _jh + "/bin:" + os.environ["PATH"]
            break
    print("JAVA_HOME:", os.environ.get("JAVA_HOME", "(not set -- install JDK 17: brew install openjdk@17)"))

# Root path by MODE
def find_root(markers=(".git", "requirements.txt", "data")):
    """Walk up from the current dir until we find the repo root."""
    here = Path.cwd().resolve()
    for cand in (here, *here.parents):
        if any((cand / m).exists() for m in markers):
            return cand
    raise FileNotFoundError("Could not locate project root (no .git/requirements.txt/data found)")

PROJECT_ROOT_BY_MODE = {
    "COLAB":   Path("/content/drive/MyDrive/DSC 288R/Project"),
    "EXPANSE": Path("/home/bguo3/bguo3/DSC-288R-Capstone-Final-Project"),
    "LOCAL":   find_root(),
}
ROOT = PROJECT_ROOT_BY_MODE[MODE.upper()]

# Add the root path to global system
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

# Print the root path
print("MODE:", MODE)
print("PROJECT_ROOT:", ROOT)

MODE: LOCAL
PROJECT_ROOT: /Users/steveg/Desktop/DSC-288R-Capstone-Final-Project


In [11]:
from pyspark.sql import functions as F

from src.utils.pyspark_utils import create_spark_session
from src.utils.paths_utils import ProjectPaths
from src.utils.io_utils import read_spark_parquet
import src.pipelines.ML_modeling as ml

import time

## Read Train/Validation/Test Full/Sampled Dataset From Parquet

In [12]:
# Set up for Spark app & resource allocation
# spark = create_spark_session("steam_reviews_machine_learning_modeling")
spark = create_spark_session(
    "steam_reviews_machine_learning_finetuning",
    extra_configs={"spark.driver.memory": "4g", "spark.master": "local[*]"},
)

# Load data
paths = ProjectPaths(MODE)
if MODE.upper() == "EXPANSE":
    train = read_spark_parquet(spark=spark, path=paths.random_row_train_parquet)
    val = read_spark_parquet(spark=spark, path=paths.random_row_val_parquet)
    test = read_spark_parquet(spark=spark, path=paths.random_row_test_parquet)
else:
    train = read_spark_parquet(spark=spark, path=paths.random_row_train_sampled_parquet)
    val = read_spark_parquet(spark=spark, path=paths.random_row_val_sampled_parquet)
    test = read_spark_parquet(spark=spark, path=paths.random_row_test_sampled_parquet)

Read Spark parquet from: /Users/steveg/Desktop/DSC-288R-Capstone-Final-Project/data/train_val_test_splits/random_row/train_sampled_parquet
Read Spark parquet from: /Users/steveg/Desktop/DSC-288R-Capstone-Final-Project/data/train_val_test_splits/random_row/val_sampled_parquet
Read Spark parquet from: /Users/steveg/Desktop/DSC-288R-Capstone-Final-Project/data/train_val_test_splits/random_row/test_sampled_parquet


## Hyperparameter Tuning with Cross-Validation

We tune each model with k-fold cross-validation through the shared
`src.pipelines.ML_modeling` (`ml`) helpers, so the estimators, evaluators, and
label handling stay identical to the baseline modeling notebook:

- `ml.cross_validate_model(...)` builds the estimator from `ml.build_models`,
  searches a param grid with `CrossValidator`, and returns the fitted CV model,
  the refit `best_model`, the winning params, and a tidy `results_df`.
- `ml.evaluate_split(...)` scores the tuned `best_model` on the held-out test
  split using the same metric definitions as `ml.evaluate_model`.

### Logistic Regression

Cross-Validation Results (Logistic Regression):
To evaluate model stability, we applied 5-fold cross-validation using Logistic Regression as a baseline classifier. The model achieved a final ROC-AUC score of 0.864 on the held-out test set. An ROC-AUC value above 0.80 is generally considered strong discrimination performance, indicating that the model is effective at distinguishing between churned and retained players across a range of classification thresholds. These results suggest that the behavioral features derived from player activity contain meaningful predictive information and that the model's performance remains robust when evaluated across multiple validation folds rather than relying on a single train-validation split.

In [13]:
start = time.time()

# Tune regularization strength + elastic-net mix via 5-fold CV.
lr_cv = ml.cross_validate_model(
    model_name="log_reg",
    train_final_df=train,
    spark=spark,
    param_grid={
        "regParam": [0.01, 0.1],
        "elasticNetParam": [0.0, 0.5],
    },
    num_folds=5,
    metric="areaUnderROC",
    parallelism=2,
    mode=MODE,
)
display(lr_cv["results_df"])

elapsed = time.time() - start
print(f"Logistic Regression CV took: {elapsed:.2f} seconds")


===== Cross-validating log_reg (4 param combo(s) x 5 folds) =====


log_reg best CV areaUnderROC: 0.8973
log_reg best params: {'regParam': 0.01, 'elasticNetParam': 0.0}


,regParam,elasticNetParam,cv_areaUnderROC
0,0.01,0.0,0.897297
1,0.01,0.5,0.896844
2,0.10,0.0,0.870945
3,0.10,0.5,0.863046


Logistic Regression CV took: 350.32 seconds


In [14]:
# Evaluate the tuned best model on the held-out test split.
lr_test_pred = lr_cv["best_model"].transform(test)

lr_test_scores = ml.evaluate_split(
    model_name="log_reg",
    pred_df=lr_test_pred,
    split_name="Test",
    metrics=["areaUnderROC"],
)
print("Final test ROC-AUC:", lr_test_scores["areaUnderROC"])

log_reg Test areaUnderROC: 0.8973
Final test ROC-AUC: 0.8972613987497932


### XGBoost

XGBoost slightly outperformed the Logistic Regression baseline, achieving a test ROC-AUC of 0.878 compared with Logistic Regression’s 0.864. This suggests that nonlinear tree-based modeling captured additional relationships in player behavior beyond the linear patterns learned by Logistic Regression. However, because the dataset is highly imbalanced, ROC-AUC and F1 are more informative than accuracy alone.

In [15]:
start = time.time()

# Empty param grid -> 5-fold CV on the baseline XGBoost params from build_models
# (label cast to int is handled inside cross_validate_model).
xgb_cv = ml.cross_validate_model(
    model_name="xgb",
    train_final_df=train,
    spark=spark,
    param_grid={},
    num_folds=5,
    metric="areaUnderROC",
    parallelism=2,
    mode=MODE,
)
print("XGBoost 5-Fold CV ROC-AUC:", xgb_cv["best_avg_metric"])

elapsed = time.time() - start
print(f"XGBoost CV took: {elapsed:.2f} seconds")


===== Cross-validating xgb (1 param combo(s) x 5 folds) =====


2026-06-04 18:58:28,359 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 7 workers with
	booster params: {'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 6, 'objective': 'binary:logistic', 'subsample': 0.8, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
2026-06-04 18:58:37,904 INFO XGBoost-PySpark: _train_booster Training on CPUs 7]
[18:58:39] Task 0 got rank 0
[18:58:39] Task 1 got rank 1
[18:58:39] Task 2 got rank 2
[18:58:39] Task 3 got rank 3
[18:58:39] Task 4 got rank 4
[18:58:39] Task 5 got rank 5
[18:58:39] Task 6 got rank 6
[18:58:40] [0]	training-logloss:0.23659
[18:58:40] [1]	training-logloss:0.23361
[18:58:40] [2]	training-logloss:0.21816
[18:58:40] [3]	training-logloss:0.17545
[18:58:40] [4]	training-logloss:0.15314
[18:58:40] [5]	training-logloss:0.13740
[18:58:40] [6]	training-logloss:0.12522
[18:58:40] [7]	training-logloss:0.11542
[18:58:40

xgb best CV areaUnderROC: 0.9924
XGBoost 5-Fold CV ROC-AUC: 0.9924353422933242
XGBoost CV took: 165.58 seconds


In [16]:
# Evaluate the tuned best model on the held-out test split.
xgb_test_pred = xgb_cv["best_model"].transform(test)

xgb_test_scores = ml.evaluate_split(
    model_name="xgb",
    pred_df=xgb_test_pred,
    split_name="Test",
    metrics=["areaUnderROC"],
)
print("XGBoost CV final test ROC-AUC:", xgb_test_scores["areaUnderROC"])

2026-06-04 19:01:11,427 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs


xgb Test areaUnderROC: 0.9923
XGBoost CV final test ROC-AUC: 0.9922893315908342


5-Fold Cross-Validation Results

To further evaluate model stability and reduce dependence on a single train-test split, we applied 5-fold cross-validation to both the Logistic Regression baseline and the XGBoost model. Cross-validation repeatedly partitions the training data into different folds, allowing each subset to serve as validation data once while the remaining folds are used for training.

The Logistic Regression baseline achieved a cross-validated ROC-AUC of 0.864, demonstrating strong ability to distinguish between churned and retained players using a linear decision boundary. XGBoost achieved a slightly higher cross-validated ROC-AUC of 0.877, indicating that the ensemble tree-based approach was able to capture additional nonlinear relationships within player behavioral features.

To verify that performance generalized beyond the validation folds, the best XGBoost cross-validation model was evaluated on a held-out test set and achieved a ROC-AUC of 0.878. The near-identical performance between the cross-validation score (0.877) and the final test score (0.878) suggests that the model generalized well and did not exhibit significant overfitting.

Overall, the cross-validation results support the conclusion that player engagement features such as playtime history, review behavior, voting activity, and community interaction metrics provide meaningful predictive signals for churn prediction. While Logistic Regression provided a strong baseline, XGBoost consistently achieved the best performance and was selected as the final model.

## Save Model to Snappy

In [17]:
# Persist the tuned best models to <root>/models/tuned/<model_name>/.
tuned_models = {
    "log_reg": lr_cv["best_model"],
    "xgb": xgb_cv["best_model"],
}
for model_name, model in tuned_models.items():
    ml.save_model(model, model_name, paths.models_root / "tuned")

Saved log_reg model to: /Users/steveg/Desktop/DSC-288R-Capstone-Final-Project/models/tuned/log_reg


Saved xgb model to: /Users/steveg/Desktop/DSC-288R-Capstone-Final-Project/models/tuned/xgb


In [18]:
spark.stop()